In [15]:
import pandas as pd
import numpy as np
from scipy.stats import qmc
from sklearn.preprocessing import StandardScaler as SS
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
import xgboost as xgb
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
from sklearn.exceptions import ConvergenceWarning
import plotly.express as px
import itertools

In [ ]:
lista_zavorra = [0, 20, 40, 60]
campioni_per_zavorra = 50

limiti_inferiori = [0.0, -4.5, 0.0, 1.5]
limiti_superiori = [6.0, -2.0, 0.3, 4.5]

risultati = []

#LHS
sampler = qmc.LatinHypercube(d=4, seed=42)

for zavorra in lista_zavorra:
    campioni_lhs = sampler.random(n=campioni_per_zavorra)
    campioni_scalati = qmc.scale(campioni_lhs, limiti_inferiori, limiti_superiori)

    for riga in campioni_scalati:
        vento = round(riga[0], 2)
        coeff_teta = round(riga[1], 2)
        std_vento = round(riga[2], 3)
        delay = round(riga[3], 2)

        teta_rampa = round(vento * coeff_teta, 2)

        risultati.append({
            'vento_medio': vento,
            'std_vento': round(std_vento*vento,2),
            'zavorra': zavorra,
            'delay_secondi': delay,
            'angolo_rampa': teta_rampa,
            'APOGEO_MISURATO': '',
            'ANGOLO_ACCENSIONE_C6': ''
        })

df = pd.DataFrame(risultati)
df.to_csv("Simulazioni_GEMINI.csv", index=False)

In [3]:
df = pd.read_csv("Simulazioni_GEMINI.csv")

#"DATA CLEANING"
for column in df.columns:
  df[column] = df[column].astype("str")
  df[column] = df[column].str.replace(",", ".")
  df[column] = df[column].str.replace(r'[^0-9.]', '', regex=True)
  df[column] = df[column].apply(lambda x: x.split('.')[0] + '.' + x.split('.')[1] if len(x.split('.')) > 2 else x)
  df[column] = df[column].astype("float")

#SHUFFLE
df = df.sample(frac=1, random_state=999).reset_index(drop=True)

#LABELS
y1 = df["APOGEO_MISURATO"]
y2 = df["ANGOLO_ACCENSIONE_C6"]

#PREDICTIVE VARIABLES
x = df.drop(["APOGEO_MISURATO", "ANGOLO_ACCENSIONE_C6"], axis=1)

#NORMALIZATION
scaler = SS()
x_norm = scaler.fit_transform(x)

In [4]:
#RANDOM FOREST
modello_rf = RandomForestRegressor(
    n_estimators=100,
    random_state=999,
    n_jobs=-1)

#XGBOOST
modello_xgb = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=999,
    n_jobs=-1)

#SUPPORT VECTOR REGRESSION
modello_svr = SVR(
    kernel='rbf',
    C=50, #100,
    gamma='scale')

#NN
nn = MLPRegressor(
  hidden_layer_sizes=(16, 8),
  activation='tanh',
  solver='lbfgs',
  alpha=0.1,
  max_iter=5000,
  tol=0.001,
  random_state=999)

modelli_livello_0 = [('m1', modello_rf), ('m2', modello_xgb), ('m3', modello_svr), ('m4', nn)]

In [5]:
#STACKING
stacking = StackingRegressor(
  estimators=modelli_livello_0,
  final_estimator=Ridge(),
  cv=5,
  n_jobs=-1)

In [24]:
#FUNZIONE PER PREVISIONI
def val(modello, dati_x, reale):
    kf = KFold(n_splits=5, shuffle=True, random_state=999)
    return cross_val_predict(modello, dati_x, reale, cv=kf)

#METRICHE MODELLI
def valuta_metriche(target, reale, previsioni, maschera):
    reale_filtrato = reale[maschera]
    previsioni_filtrate = previsioni[maschera]
    
    mae = mean_absolute_error(reale_filtrato, previsioni_filtrate)
    rmse = np.sqrt(mean_squared_error(reale_filtrato, previsioni_filtrate))
    
    print(f"Risultati modello x previsione {target} (su {len(reale_filtrato)} lanci approvati)")
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}\n")

In [25]:
print("Cucinando...\n")

#PREVISIONI
prev_apo = val(stacking, x_norm, y1)
prev_ang = val(stacking, x_norm, y2)

#MASCHERA
valore_maschera = 75
maschera = prev_ang > valore_maschera 

#VALUTAZIONE (Passiamo LA STESSA maschera a entrambi)
valuta_metriche("APOGEO", y1, prev_apo, maschera)
valuta_metriche("ANGOLO", y2, prev_ang, maschera)

Cucinando...

Risultati modello x previsione APOGEO (su 119 lanci approvati)
MAE: 4.09
RMSE: 5.77

Risultati modello x previsione ANGOLO (su 119 lanci approvati)
MAE: 5.96
RMSE: 8.17



In [26]:
#ANALISI DEI RESIDUI PER IL MODELLO DI PREVISIONE DELL'ANGOLO
maschera = prev_ang > valore_maschera

reale_ang_filtrato = y2[maschera]
prev_ang_filtrato = prev_ang[maschera]

residui_ang = reale_ang_filtrato - prev_ang_filtrato
df_residui = pd.DataFrame({
    'Reale': reale_ang_filtrato,
    'Previsto': prev_ang_filtrato,
    'Residuo': residui_ang})

fig = px.scatter(
    df_residui, 
    x='Previsto', 
    y='Residuo',
    title='Analisi dei Residui: Angolo alla Partenza del Secondo Motore (Finestra Operativa > 75°)',
    labels={
        'Previsto': 'Angolo Previsto dal Modello (°)', 
        'Residuo': 'Errore Residuo (Reale - Previsto) (°)'
    },
    template='plotly_white',
    hover_data=['Reale'])

fig.add_hline(y=0, line_dash="dash", line_color="red", line_width=2)
fig.show()

In [27]:
warnings.filterwarnings("ignore", category=ConvergenceWarning)

#ADDENSTRAMENTO FINALE
modello_apogeo = StackingRegressor(estimators=modelli_livello_0, final_estimator=Ridge(), cv=5)
modello_angolo = StackingRegressor(estimators=modelli_livello_0, final_estimator=Ridge(), cv=5)
modello_apogeo.fit(x_norm, y1)
modello_angolo.fit(x_norm, y2)

#SALVATAGGIO MODELLI
joblib.dump(modello_apogeo, "modello_apogeo.pkl")
joblib.dump(modello_angolo, "modello_angolo.pkl")
joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']

In [28]:
#CARICAMENTO MODELLI --> RIPARTIRE DA QUI SE I MODELLI SONO GIA' STATI ADDENSTRATI
modello_apo = joblib.load('modello_apogeo.pkl')
modello_ang = joblib.load('modello_angolo.pkl')
scaler_fisso = joblib.load('scaler.pkl')